# Inteligencia Artificial — Práctica de Laboratorio

## Implementación Manual de K-Nearest Neighbors (KNN) - Nivel Principiante

---

### Información General

| Campo | Detalle |
|---|---|
| **Estudiante** | Alex Guaman |
| **Asignatura** | Inteligencia Artificial / Machine Learning |
| **Tema** | Algoritmo K-Nearest Neighbors (KNN) de forma sencilla |
| **Dataset** | Pacientes Ecuatorianos |

---

### 1. Cargar los Datos
Primero, traemos la información del archivo CSV usando la librería `pandas`.

In [ ]:
import pandas as pd
import math

# Leemos el archivo
df = pd.read_csv('../data/pacientes.csv')
df

,sexo,ciudad,colesterol,edad,diabetes
0,1,Cuenca,bajo,18,no
1,2,Quito,alto,52,si
2,2,Guayaquil,medio,34,no
3,1,Loja,alto,61,si
4,2,Ambato,medio,45,no
5,1,Machala,muy alto,67,si


### 2. Identificar las Variables
Antes de empezar, debemos saber qué tipo de datos tenemos:

| Variable | Tipo |
|---|---|
| `sexo` | Categoría (1 o 2) |
| `ciudad` | Nombre de la ciudad |
| `colesterol` | Nivel (bajo, medio, alto, muy alto) |
| `edad` | Número |
| `diabetes` | **Objetivo** (lo que queremos predecir) |

### 3. Transformar los datos a números
Las computadoras solo entienden números, así que cambiamos las palabras por valores numéricos.

In [ ]:
df['ciudad_num'] = df['ciudad'].replace({
    'Cuenca': 0, 'Quito': 1, 'Guayaquil': 2, 'Loja': 3, 'Ambato': 4, 'Machala': 5
})

df['colesterol_num'] = df['colesterol'].replace({
    'bajo': 1, 'medio': 2, 'alto': 3, 'muy alto': 4
})

df['diabetes_num'] = df['diabetes'].replace({'no': 0, 'si': 1})

columnas_prediccion = ['sexo', 'ciudad_num', 'colesterol_num', 'edad']
X = df[columnas_prediccion]
Y = df['diabetes_num']

print("Datos transformados:")
X

Datos transformados:


,sexo,ciudad_num,colesterol_num,edad
0,1,0,1,18
1,2,1,3,52
2,2,2,2,34
3,1,3,3,61
4,2,4,2,45
5,1,5,4,67


### 4. Estandarización (Poner todo en la misma escala)
Como la `edad` llega hasta 67 y el `colesterol` solo hasta 4, debemos equilibrarlos para que el algoritmo no se confunda.

In [ ]:
medias = X.mean()
desviaciones = X.std()

X_escalado = (X - medias) / desviaciones

print("Datos escalados (estandarizados):")
X_escalado

Datos escalados (estandarizados):


,sexo,ciudad_num,colesterol_num,edad
0,-0.912871,-1.336306,-1.430194,-1.559609
1,0.912871,-0.801784,0.476731,0.322996
2,0.912871,-0.267261,-0.476731,-0.673677
3,-0.912871,0.267261,0.476731,0.821332
4,0.912871,0.801784,-0.476731,-0.064599
5,-0.912871,1.336306,1.430194,1.153557


### 5. Algoritmo KNN Manual
Creamos una función para predecir si un nuevo paciente tiene diabetes.

In [ ]:
def predecir_diabetes(nuevo_sexo, nueva_ciudad, nuevo_colesterol, nueva_edad):
    # 1. Transformar los datos del nuevo paciente a números
    ciudad_n = {'Cuenca': 0, 'Quito': 1, 'Guayaquil': 2, 'Loja': 3, 'Ambato': 4, 'Machala': 5}[nueva_ciudad]
    colesterol_n = {'bajo': 1, 'medio': 2, 'alto': 3, 'muy alto': 4}[nuevo_colesterol]
    
    nuevo_p = [nuevo_sexo, ciudad_n, colesterol_n, nueva_edad]
    
    # 2. Escalar el nuevo paciente usando las mismas medias y desviaciones
    nuevo_p_escalado = []
    for i in range(4):
        valor_esc = (nuevo_p[i] - medias.iloc[i]) / desviaciones.iloc[i]
        nuevo_p_escalado.append(valor_esc)
    
    # 3. Calcular distancias con todos los pacientes del dataset
    todas_las_distancias = []
    
    for i in range(len(X_escalado)):
        paciente_entrenamiento = X_escalado.iloc[i]
        
        # Fórmula de Distancia Euclídea
        suma_cuadrados = 0
        for j in range(4):
            suma_cuadrados += (nuevo_p_escalado[j] - paciente_entrenamiento.iloc[j])**2
        
        distancia = math.sqrt(suma_cuadrados)
        todas_las_distancias.append( (distancia, Y.iloc[i]) )
    
    # 4. Ordenar por distancia y elegir los 3 más cercanos
    todas_las_distancias.sort()
    vecinos = todas_las_distancias[:3]
    
    # 5. Contar votos (cuántos tienen diabetes y cuántos no)
    votos_si = 0
    votos_no = 0
    
    print("--- Vecinos Seleccionados ---")
    for d, clase in vecinos:
        print(f"Distancia: {d:.4f} | Diabetes: {'si' if clase == 1 else 'no'}")
        if clase == 1:
            votos_si += 1
        else:
            votos_no += 1
            
    # Resultado final
    if votos_si > votos_no:
        return "SÍ tiene diabetes"
    else:
        return "NO tiene diabetes"

# Probamos con el nuevo paciente
resultado = predecir_diabetes(nuevo_sexo=2, nueva_ciudad='Cuenca', nuevo_colesterol='alto', nueva_edad=50)


---

### 6. Conclusiones

1.  **Importancia del Preprocesamiento:** Se demostró que transformar datos categóricos (como ciudad y colesterol) a números es fundamental para que el algoritmo pueda realizar cálculos matemáticos.
2.  **Efecto de la Estandarización:** Sin el escalado (Z-score), variables con rangos grandes como la `edad` dominarían el cálculo de la distancia, ignorando otras variables importantes. Al estandarizar, todas las variables tienen el mismo peso.
3.  **Lógica del Algoritmo KNN:** El modelo predijo que el paciente **NO tiene diabetes** basándose en que 2 de sus 3 vecinos más cercanos tampoco tenían la enfermedad.
4.  **Aprendizaje Manual:** Implementar el algoritmo paso a paso permite entender que detrás de las librerías complejas solo hay matemáticas básicas como la distancia euclídea y votaciones por mayoría.